# Sketch2Shape on Kaggle

Turns a sketch (drawn or uploaded) into a textured 3D model (`.glb` / `.obj`) in a web app.

**Before you start**
1. Right panel → **Session options**: Accelerator **GPU T4 x2**, Internet **On** (a phone-verified Kaggle account is needed for GPUs).
2. Optional: **Add-ons → Secrets**, add `OPENAI_API_KEY` (or `ANTHROPIC_API_KEY`) and switch it on for this notebook. It lets a vision LLM name the sketch and check the generated image; without a key the app still works. `HF_TOKEN` is optional (faster downloads).

**Run cells 1 → 4 in order.** Cell 4 prints the public link (`https://….gradio.live`) of the app.
- Cell 3 (setup) takes about 15-25 minutes on a fresh session (installs, two CUDA extensions, ~20 GB of model weights). Rerunning it later is quick.
- The first model after the app starts takes ~5 minutes while the models load; later ones about 3-4 minutes.

Everything after cell 4 is optional (logs, stopping the app, tests, a batch run on the example sketches).

In [ ]:
# 1. Check the GPUs and get the code
import os, subprocess, sys

ROOT = "/kaggle/working/sketch2shape"
REPO_URL = "https://github.com/RohanSaikumar/sketch2shape.git"

try:
    gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpus = ""
print(gpus or "No GPU found")
if gpus.count("GPU ") == 0:
    raise SystemExit("No GPU: right panel → Session options → Accelerator 'GPU T4 x2'. "
                     "The session restarts; then run this cell again.")
if gpus.count("GPU ") == 1:
    print("One GPU: every model runs on it, one after another (slower than T4 x2, but it works).")

if os.path.exists(f"{ROOT}/.git"):
    step = subprocess.run(["git", "-C", ROOT, "pull", "--ff-only"], capture_output=True, text=True)
else:
    step = subprocess.run(["git", "clone", REPO_URL, ROOT], capture_output=True, text=True)
print(step.stdout.strip() or step.stderr.strip())
if step.returncode != 0:
    raise SystemExit("Could not get the code. Check that Internet is On in Session options, then run this cell again.")
os.chdir(ROOT)
print("Code ready in", ROOT, "at commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                                         capture_output=True, text=True).stdout.strip())

In [ ]:
# 2. Optional API keys from Kaggle Secrets (skip freely: everything works without them)
import os
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
except ImportError:
    secrets = None
    print("Not on Kaggle: set keys as environment variables if you want them.")
for name in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "HF_TOKEN"):
    if secrets is None:
        break
    try:
        os.environ[name] = secrets.get_secret(name)
        print(name, "loaded")
    except Exception:
        print(name, "not set (optional)")

In [ ]:
# 3. One-time setup (15-25 min on a fresh session; quick when rerun): Hunyuan3D-2 at a pinned commit,
#    its two CUDA extensions, one compatibility patch, this project's requirements, the model weights.
#    Progress lines start with "==". The full log is /kaggle/working/setup.log.
import os, subprocess
ROOT = "/kaggle/working/sketch2shape"
os.chdir(ROOT)
with open("/kaggle/working/setup.log", "w") as log:
    setup = subprocess.Popen(["bash", "scripts/setup.sh"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in setup.stdout:
        log.write(line)
        if line.startswith("==") or line.startswith("patch"):
            print(line.rstrip(), flush=True)
    setup.wait()
if setup.returncode != 0:
    print(open("/kaggle/working/setup.log").read()[-4000:])
    raise SystemExit("Setup failed; the end of the log is above (full log: /kaggle/working/setup.log).")
print("Setup finished. Lines like \"ERROR: pip's dependency resolver ...\" in the log are warnings and can be ignored.")

In [ ]:
# 4. Start the web app and print its public link (about a minute). Rerun this cell to restart the app (new link).
import os, re, subprocess, sys, time
ROOT = "/kaggle/working/sketch2shape"
os.chdir(ROOT)
if not os.environ.get("OPENAI_API_KEY") and not os.environ.get("ANTHROPIC_API_KEY"):
    try:                                    # keys from cell 2 are lost if the kernel restarted
        from kaggle_secrets import UserSecretsClient
        for name in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "HF_TOKEN"):
            try:
                os.environ[name] = UserSecretsClient().get_secret(name)
            except Exception:
                pass
    except ImportError:
        pass

subprocess.run(["pkill", "-f", "app[.]py"])                 # stop an app started earlier
time.sleep(2)
log_path = "/kaggle/working/app.log"
app = subprocess.Popen([sys.executable, "app.py", "--share"], stdout=open(log_path, "w"),
                       stderr=subprocess.STDOUT, start_new_session=True)
link, text = None, ""
for _ in range(100):                                        # wait up to ~5 min for the link
    time.sleep(3)
    text = open(log_path, encoding="utf-8", errors="replace").read()
    found = re.findall(r"https://[\w.-]+\.gradio\.live", text)
    if found:
        link = found[-1]
        break
    if app.poll() is not None:
        break
if link:
    print("Open the app:", link)
    print("Draw or upload a sketch, optionally describe it, and click 'Make 3D model'.")
    print("Wait for 'SDXL ready · 3D shape model ready' under the progress bar for the fastest first model.")
else:
    print(text[-3000:])
    print("\nNo link yet: the end of the app log is above. Run this cell again; if it repeats, see cell 5.")

## Optional

- **Cell 5**: the app's recent log (if the link does not open or a run fails).
- **Cell 6**: stop the app (frees both GPUs). Needed before cell 8.
- **Cell 7**: offline tests (no GPU needed, about a minute).
- **Cell 8**: run every example sketch in `examples/sketches` and show the results.
- **Cell 9**: results table and picture strips from the app's own runs (for reports).

Cells 6-8 only act when you set the switch at their top to `True`, so "Run All" ends with the app running.

In [ ]:
# 5. The app's recent log
import os
log_path = "/kaggle/working/app.log"
print(open(log_path, encoding="utf-8", errors="replace").read()[-4000:] if os.path.exists(log_path)
      else "No app log yet: run cell 4 first.")

In [ ]:
# 6. Stop the app and its model workers (frees both GPUs). Off by default, so "Run All" keeps the app running.
import subprocess
STOP_APP = False                     # set to True, then run this cell
if STOP_APP:
    subprocess.run(["pkill", "-f", "app[.]py"])
    subprocess.run(["pkill", "-9", "-f", "sketch2shape[.]worker"])
    print("App stopped. Run cell 4 to start it again.")
else:
    print("Skipped (STOP_APP = False).")

In [ ]:
# 7. Offline tests (no GPU needed, about a minute). Off by default.
import subprocess
RUN_TESTS = False                    # set to True, then run this cell
if RUN_TESTS:
    print(subprocess.run(["python", "-m", "pytest", "-q", "tests"], cwd="/kaggle/working/sketch2shape",
                         capture_output=True, text=True).stdout[-3000:])
else:
    print("Skipped (RUN_TESTS = False).")

In [ ]:
# 8. Run every example sketch and show the results. Stop the app first (cell 6): both need the GPUs.
#    About 5 min per sketch. Writes results/results.md, results/<sketch>.glb and results/sheets/<sketch>.png
import json, os, subprocess, sys
from IPython.display import Markdown, display
from PIL import Image
RUN_EXAMPLES = False                 # set to True, then run this cell
if RUN_EXAMPLES:
    os.chdir("/kaggle/working/sketch2shape")
    batch = subprocess.Popen([sys.executable, "-u", "scripts/eval_batch.py", "examples/sketches", "--prompts",
                              "examples/prompts.json", "--out", "results"], stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
    for line in batch.stdout:
        if line.startswith(("==", "{", "|", "Successful")) or "Error" in line:
            print(line.rstrip(), flush=True)
    batch.wait()
    for r in json.load(open("results/results.json")):
        display(Markdown(f"**{r['sketch']}**: `{r['object']}`, texture: {r['texture']}, {r['seconds']} s, ok={r['ok']}"))
        sheet = f"results/sheets/{r['sketch'].rsplit('.', 1)[0]}.png"
        if os.path.exists(sheet):
            display(Image.open(sheet))
else:
    print("Skipped (RUN_EXAMPLES = False).")

In [ ]:
# 9. Results table and picture strips from the app's own runs (newest 8); safe while the app runs
import glob, os, subprocess, sys
from IPython.display import display
from PIL import Image
os.chdir("/kaggle/working/sketch2shape")
print(subprocess.run([sys.executable, "scripts/collect_results.py", "--last", "8"], capture_output=True, text=True).stdout)
for sheet in sorted(glob.glob("results/sheets/2*.png"), reverse=True)[:8]:
    display(Image.open(sheet))